In [1]:
import sys
sys.path.append("../IMAGE-Mat/scripts/vema")
from preprocessing import preprocessing


In [2]:
import prism

In [3]:
base_dir = "../IMAGE-Mat/scripts/vema"
preprocessing_results = preprocessing(base_dir)

/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-

In [4]:
vehicle_nr = preprocessing_results['total_nr_vehicles_simple']
life_time_vehicles = preprocessing_results["lifetimes_vehicles"].to_xarray()

In [5]:
life_time_vehicles

<xarray.Dataset> Size: 75kB
Dimensions:                          (year: 254)
Coordinates:
  * year                             (year) int64 2kB 1807 1808 ... 2059 2060
Data variables: (12/36)
    ('air_pas', 'mean')              (year) float64 2kB 20.0 20.0 ... 22.0 22.0
    ('air_freight', 'mean')          (year) float64 2kB 21.0 21.0 ... 23.1 23.1
    ('rail_reg', 'mean')             (year) float64 2kB 35.0 35.0 ... 49.0 49.0
    ('rail_hst', 'mean')             (year) float64 2kB 30.0 30.0 ... 42.0 42.0
    ('rail_freight', 'mean')         (year) float64 2kB 38.0 38.0 ... 51.3 51.3
    ('sea_shipping_small', 'mean')   (year) float64 2kB 26.0 26.0 ... 34.67
    ...                               ...
    ('reg_bus', 'stdev')             (year) float64 2kB 0.322 0.322 ... 0.322
    ('inland_shipping', 'stdev')     (year) float64 2kB 0.266 0.266 ... 0.266
    ('bicycle', 'stdev')             (year) float64 2kB 0.266 0.266 ... 0.266
    ('car', 'stdev')                 (year) float64 2kB 0.4 0.4 0.4 ... 0.4 0.4
    ('car', 'shape')                 (year) float64 2kB 2.01 2.01 ... 2.01 2.01
    ('car', 'scale')                 (year) float64 2kB 16.07 16.07 ... 20.38

In [6]:
from survival import ScipySurvival, convert_life_time_vehicles, SurvivalMatrix
from util import convert_vehicles
from stock import compute_historic, compute_dynamic_stock_driven
import numpy as np
       
lifetime_params = convert_life_time_vehicles(life_time_vehicles)
survival = ScipySurvival(lifetime_params)
survival_matrix = SurvivalMatrix(survival)
xr_vehicle_nr = convert_vehicles(vehicle_nr)

In [7]:
start_simulation = 1970
end_simulation = xr_vehicle_nr.coords["time"].max()
Region = prism.NewDim("region", coords=[str(x) for x in np.arange(1, 27)])
Mode = prism.NewDim("mode", coords=list(vehicle_nr.columns.levels[0].unique()))
Cohort = prism.NewDim("cohort", coords=[str(x) for x in vehicle_nr.index.to_numpy()])

In [8]:
@prism.interface
class Stocks(prism.Model):
    start_simulation: int
    survival_matrix: SurvivalMatrix
    stock: prism.TimeVariable[Region, Mode] #TODO check how to have property that can be both input and output within prism
    stock_function: function    # defines the stock function to use e.g. stock or inflow driven

    stock_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0)) 
    inflow: prism.TimeVariable[Region, Mode, "count"]         = prism.export(initial_value = prism.Array[Region, Mode, 'count'](0.0))   
    outflow_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0))

    def compute_initial_values(self, time: prism.Timeline):
        compute_historic(self.stock, self.survival_matrix, self.start_simulation,
                         self.stock_by_cohort, self.inflow, self.outflow_by_cohort,
                         self.stock_function)

    def compute_values(self, time: prism.Time):
        t, dt = time.t, time.dt
        self.stock_function(self.stock, self.survival_matrix, self.start_simulation,
                            self.stock_by_cohort, self.inflow, self.outflow_by_cohort)

In [9]:
timeline = prism.Timeline(start=xr_vehicle_nr.coords["time"][0],
                          end=end_simulation, stepsize=1)
timeline_simulate = prism.Timeline(start=start_simulation,
                          end=end_simulation, stepsize=1)

In [10]:
model = Stocks(timeline, start_simulation=start_simulation, survival_matrix=survival_matrix, stock=xr_vehicle_nr, stock_function = compute_dynamic_stock_driven)

In [11]:
model.simulate(timeline_simulate)

Coordinates:
  * mode     (mode) <U25 2kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'
  * region   (region) <U21 0B 
    time     int64 8B 1808 Coordinates:
    time     int64 8B 1808
    cohort   int64 8B 1808
  * mode     (mode) <U18 1kB 'car' 'air_pas' ... 'inland_shipping' 'bicycle' <xarray.DataArray 'time' ()> Size: 8B
array(1808)
Coordinates:
    time     int64 8B 1808


ValueError: cannot align objects with join='exact' where index/labels/sizes are not equal along these coordinates (dimensions): 'mode' ('mode',)